# Face-aging inference API guide

This notebook is a compact guide to the project inference functions. Use **fixed inference** for controlled experiments, **adaptive inference** when edit strength should grow with `|target_age - source_age|`, **age sweeps** for visual inspection, and **checkpoint diagnostics** when MiVOLO/ArcFace tables and CSVs are required. Run it in the existing `deep_learning` Conda environment.

In [ ]:
from pathlib import Path

import torch
from IPython.display import display
from PIL import Image

from src.inference import (
    DEFAULT_STRENGTH_MAP,
    compare_inference_modes,
    diagnose_checkpoint_adaptive_age_sweep,
    diagnose_checkpoint_age_sweep,
    diagnose_checkpoint_smart_age_sweep,
    diagnose_checkpoint_strength_sweep,
    generate_adaptive_age_sweep,
    generate_age_sweep,
    generate_aged_face_adaptive_strength,
    infer_face_aging,
    load_face_aging_inference_bundle,
    run_inference_pipeline_validation,
    save_inference_image,
)

## 1. Shared configuration

All examples reuse one checkpoint, source photograph and seed. `SOURCE_AGE` must be the real age in the source image. Keep `text_reference_mode='source_age'` for the referenced-CFG baseline learned during training.

In [ ]:
CHECKPOINT_PATH = Path('/server/checkpoints/face_aging/best/adapter_inference.pt')
SOURCE_IMAGE = Path('/server/images/person.jpg')
OUTPUT_DIR = Path('/server/results/face_aging')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_AGE = 26
TARGET_AGE = 65
TARGET_AGES = [8, 12, 26, 35, 45, 55, 65]
IMAGE_SIZE = 400
SEED = 2026

# Keys are inclusive upper bounds for absolute requested delta.
STRENGTH_MAP = {3: 0.18, 8: 0.24, 15: 0.30, 25: 0.36, 30: 0.40, 999: 0.44}
# Optional exact lookup by requested target age (not by delta).
TARGET_AGE_STRENGTH_MAP = {8: 0.45, 12: 0.40, 26: 0.05, 35: 0.23, 45: 0.37, 55: 0.45, 65: 0.45}
SMART_INITIAL_STRENGTH_MAP = {8: 0.35, 12: 0.34, 26: 0.04, 35: 0.18, 45: 0.27, 55: 0.27, 65: 0.25}
DELTA_BIN_THRESHOLDS = [5, 15, 30]
print('Library default:', DEFAULT_STRENGTH_MAP)

## 2. Load the checkpoint bundle

The loader reconstructs frozen SD1.5 and restores the trained adapter, expanded `conv_in`, and age conditioner. Auxiliary models are enabled because checkpoint diagnostics need ArcFace and MiVOLO; omit them for generation-only inference to save memory.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DTYPE = (
    torch.bfloat16 if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
    else torch.float16 if DEVICE.type == 'cuda'
    else torch.float32
)

bundle = load_face_aging_inference_bundle(
    CHECKPOINT_PATH,
    device=DEVICE,
    dtype=MODEL_DTYPE,
    local_files_only=False,
    load_auxiliary_models=True,
    auxiliary_dtype=torch.float32,
    auxiliary_trust_remote_code=True,  # review and pin MiVOLO code in production
)
bundle['inference_checkpoint_report']

## 3. Fixed-strength inference

`infer_face_aging` is the unchanged general API. Use it when every request must use the same strength, or when comparing direct img2img with DDIM inversion.

In [ ]:
fixed_result = infer_face_aging(
    bundle=bundle, image=SOURCE_IMAGE,
    source_age=SOURCE_AGE, target_age=TARGET_AGE,
    mode='direct', use_inverse_diffusion=False,
    strength=0.35, num_inference_steps=50,
    text_reference_mode='source_age', age_guidance_scale=3.0,
    text_guidance_scale=7.0, image_guidance_scale=1.5,
    seed=SEED, image_size=IMAGE_SIZE, compute_diagnostics=True,
)
fixed_path = save_inference_image(fixed_result, OUTPUT_DIR / 'fixed_strength.png')
print(f'Saved: {fixed_path} | effective strength: {fixed_result["strength"]:.2f}')
print(fixed_result['diagnostics'])
display(fixed_result['image'])

## 4. Adaptive-strength inference

`generate_aged_face_adaptive_strength` is direct img2img with one difference: it resolves strength from the absolute requested delta. Threshold keys are sorted internally, both aging and rejuvenation use the same magnitude rule, and deltas above the largest key reuse its strength. For a known evaluation protocol, pass `target_age_strength_map` to select strength by the exact requested age instead. Exact-age mode takes priority and raises a clear error if an age is missing.

In [ ]:
adaptive_result = generate_aged_face_adaptive_strength(
    bundle=bundle, image=SOURCE_IMAGE,
    source_age=SOURCE_AGE, target_age=TARGET_AGE,
    strength_map=STRENGTH_MAP,
    target_age_strength_map=None,  # default: use the absolute-delta map above
    num_inference_steps=50,
    text_reference_mode='source_age', age_guidance_scale=3.0,
    text_guidance_scale=7.0, image_guidance_scale=1.5,
    seed=SEED, image_size=IMAGE_SIZE, compute_diagnostics=True,
)
adaptive_path = save_inference_image(adaptive_result, OUTPUT_DIR / 'adaptive_strength.png')
print(f'Saved: {adaptive_path}')
print(f'Requested delta: {TARGET_AGE - SOURCE_AGE:+d} years')
print(f'Effective strength: {adaptive_result["effective_strength"]:.2f}')
display(adaptive_result['image'])

### Exact target-age strength

Use this variant only when the requested evaluation ages are known in advance. Here requesting age 65 uses `0.45` regardless of the source age or resulting delta.

In [ ]:
exact_target_result = generate_aged_face_adaptive_strength(
    bundle=bundle, image=SOURCE_IMAGE,
    source_age=SOURCE_AGE, target_age=TARGET_AGE,
    target_age_strength_map=TARGET_AGE_STRENGTH_MAP,
    num_inference_steps=50, text_reference_mode='source_age',
    age_guidance_scale=3.0, text_guidance_scale=7.0,
    image_guidance_scale=1.5, seed=SEED, image_size=IMAGE_SIZE,
)
print('Policy:', exact_target_result['metadata']['adaptive_strength_policy'])
print('Effective strength:', exact_target_result['effective_strength'])
display(exact_target_result['image'])

## 5. Lightweight adaptive sweep

This helper uses the bundle already in memory and generates exactly one image per target. The mapped strength appears in every diagnostic label. Use `generate_age_sweep` instead when all targets must share one fixed strength.

In [ ]:
adaptive_sweep = generate_adaptive_age_sweep(
    bundle=bundle, image=SOURCE_IMAGE, ages=TARGET_AGES,
    source_age=SOURCE_AGE, strength_map=STRENGTH_MAP,
    target_age_strength_map=None,  # set an exact map only if every TARGET_AGES value is present
    output_path=OUTPUT_DIR / 'adaptive_age_sweep.png',
    annotate_diagnostics=True, include_source=True,
    num_inference_steps=50, text_reference_mode='source_age',
    age_guidance_scale=3.0, text_guidance_scale=7.0,
    image_guidance_scale=1.5, seed=SEED, image_size=IMAGE_SIZE,
)
print([result['effective_strength'] for result in adaptive_sweep['results']])
display(adaptive_sweep['grid'])

## 6. Adaptive checkpoint diagnostic

This is the main scientific evaluation wrapper. It reloads the supplied `.pt`, records effective strength per target, and writes the sample and delta-bin diagnostics. With `generate_assisted_prompt_variant=True`, it saves a base strip, an artifact-suppression prompt strip, and a vertically stacked comparison. `prompt_assistance_config=None` uses the concise defaults; supplying term lists under the same config keys replaces those defaults.

In [ ]:
diagnostic_df = diagnose_checkpoint_adaptive_age_sweep(
    checkpoint_path=CHECKPOINT_PATH, bundle=bundle,
    source_image=SOURCE_IMAGE, source_age=SOURCE_AGE,
    target_ages=TARGET_AGES, strength_map=STRENGTH_MAP,
    target_age_strength_map=None,  # or TARGET_AGE_STRENGTH_MAP for an exact-age protocol
    delta_bin_thresholds=DELTA_BIN_THRESHOLDS,
    output_dir=OUTPUT_DIR / 'adaptive_checkpoint_diagnostic',
    num_inference_steps=50, text_reference_mode='source_age',
    age_guidance_scale=3.0, text_guidance_scale=7.0,
    image_guidance_scale=1.5, seed=SEED, image_size=IMAGE_SIZE,
    generate_assisted_prompt_variant=True,
    prompt_assistance_config=None,  # or {'positive_global_terms': [...], 'negative_global_terms': [...]}
    source_mouth_state='auto',  # auto falls back safely to unknown; closed/visible_teeth are manual options
    source_expression_state='auto',  # neutral/smiling/unknown are manual options
)
print(diagnostic_df.to_string(index=False))
print('Grid:', diagnostic_df.attrs['grid_path'])
print('Samples CSV:', diagnostic_df.attrs['csv_path'])
print('Delta-bin CSV:', diagnostic_df.attrs['delta_bin_csv_path'])
display(Image.open(diagnostic_df.attrs.get('comparison_grid_path') or diagnostic_df.attrs['grid_path']))

## 7. Bias-aware smart sweep

This deterministic screening wrapper starts from the target-specific prior and tries at most five nearby strengths per age. MiVOLO is **not treated as ground truth**: its expected response uses the longitudinal calibration, and predictions inside an adaptive confidence band are considered practically equivalent so ArcFace can favor identity. When prompt assistance is enabled, the expensive search runs only for the base variant; the assisted strip performs exactly one extra inference per target with the selected strength and guidance. Set `save_all_trials=True` only when the additional base-candidate grid and trial PNGs are needed.

In [ ]:
smart_df = diagnose_checkpoint_smart_age_sweep(
    checkpoint_path=CHECKPOINT_PATH, bundle=bundle,
    source_image=SOURCE_IMAGE, source_age=SOURCE_AGE, target_ages=TARGET_AGES,
    output_dir=OUTPUT_DIR / 'smart_checkpoint_diagnostic',
    target_age_strength_map=SMART_INITIAL_STRENGTH_MAP,
    max_trials_per_target=5,
    use_bias_corrected_mivolo_target=True,
    mivolo_bias_alpha=-3.19, mivolo_bias_beta=0.841,
    mivolo_small_delta_confidence_years=4.0,
    mivolo_large_delta_confidence_years=2.0,
    mivolo_full_confidence_delta=25.0,
    age_tolerance_years=2.0,
    min_strength=0.04, max_strength=0.45,
    strength_step_coarse=0.05, strength_step_medium=0.03, strength_step_fine=0.02,
    enable_guidance_micro_search=False,
    age_guidance_scale=3.0, text_guidance_scale=7.0, image_guidance_scale=1.5,
    identity_tiebreak=True, identity_margin_for_tiebreak=1.0,
    save_all_trials=False, seed=SEED, num_inference_steps=50, image_size=IMAGE_SIZE,
    generate_assisted_prompt_variant=True,
    prompt_assistance_config=None,  # custom lists replace the built-in lists
    source_mouth_state='auto', source_expression_state='auto',
)
print(smart_df[['target_age', 'trials_run', 'strength', 'pred_age', 'expected_mivolo_target_age', 'confidence_margin_years', 'identity_cosine']].to_string(index=False))
print('Final grid:', smart_df.attrs['grid_path'])
print('Trials CSV:', smart_df.attrs['trials_csv_path'])
print('Summary CSV:', smart_df.attrs['summary_csv_path'])
display(Image.open(smart_df.attrs.get('comparison_grid_path') or smart_df.attrs['grid_path']))

## 8. Existing comparison APIs

The fixed checkpoint sweep and multi-strength sweep remain unchanged. Use the first for a fixed baseline, the second to compare several strengths in one grid, and `compare_inference_modes` to contrast direct editing with inversion.

In [ ]:
# fixed_df = diagnose_checkpoint_age_sweep(
#     CHECKPOINT_PATH, bundle, SOURCE_IMAGE, SOURCE_AGE, TARGET_AGES,
#     output_dir=OUTPUT_DIR / 'fixed_diagnostic', strength=0.35,
#     num_inference_steps=50, image_size=IMAGE_SIZE,
# )

# strength_df = diagnose_checkpoint_strength_sweep(
#     CHECKPOINT_PATH, bundle, SOURCE_IMAGE, SOURCE_AGE, TARGET_AGES,
#     strengths=[0.20, 0.27, 0.35, 0.40],
#     output_dir=OUTPUT_DIR / 'strength_diagnostic',
#     num_inference_steps=50, image_size=IMAGE_SIZE,
# )

# comparison = compare_inference_modes(
#     bundle=bundle, image=SOURCE_IMAGE, source_age=SOURCE_AGE, target_age=TARGET_AGE,
#     strength=0.35, num_inference_steps=50, seed=SEED, image_size=IMAGE_SIZE,
#     output_path=OUTPUT_DIR / 'source_direct_inverse.png',
# )
# display(comparison['grid'])

## 9. Numerical preflight

This lightweight validation checks prompt construction and analytical guidance behavior. It does not replace a real GPU checkpoint smoke test.

In [ ]:
run_inference_pipeline_validation()